Currently, the AIA region selection method includes starting from the first NuSTAR region of the day, applying a by-eye shift, and then using that region for the whole day's observations. Solar rotation will cause the ARs to move within these regions. Here we are doing a sanity check that this is ok for all the regions. 


Update that the above was apparently NOT the method at the time of the original NCCS AIA analysis. Those regions had per-orbit corrections. This reduces the problem significantly, as most of the lengthy observations on a given day were included in that part of the sample. 

In [ ]:
#Path to top-level do-dem directory - edit for your system.
path_to_dodem = '/Users/jmdunca2/do-dem/'
from sys import path as sys_path
sys_path.append(path_to_dodem+'/dodem/')

# #import nustar_dem_prep as nu
import images_and_coalignment as iac

import pickle
from astropy import units as u
from matplotlib import pyplot as plt
import numpy as np


file='/Users/jmdunca2/do-dem/reference_files/all_targets_postghost_postshut.pickle'


with open(file, 'rb') as f:
    all_targets = pickle.load(f)

In [ ]:
solar_rotation_period=(27*u.d).to(u.hr)
disk_width = 2000*u.arcsec
halfway=solar_rotation_period/2
avg_speed = disk_width/halfway
print(avg_speed)

In [ ]:
#excluding:
exclude = ['22-apr-16_1', #failed TIS
          '24-feb-22', #ghost/flare excluded
          '25-feb-22'#ghost/flare excluded
          ]

for kk in all_targets.keys():
    if not kk in exclude:
        for ll in all_targets[kk]['loc']:
            if ll == 'disk':
                #print('disk region in: ', kk)
                if len(all_targets[kk]['datapaths']) > 2:
                    print('disk region, long obs in: ', kk, len(all_targets[kk]['datapaths']), all_targets[kk]['method'])

In [ ]:
#where is the AIA analysis for each of these regions:

nccs_aia_cases = ['29-may-18_1', '09-sep-18', '10-sep-18', '12-apr-19', '13-apr-19', '29-jan-20', '29-apr-21']
jsoc_aia_cases = ['07-sep-18', '14-jan-21']

In [ ]:
import glob

for nac in nccs_aia_cases:
    print(nac)
    ARDict = all_targets[nac]
    #print(ARDict.keys())
    aiadir = ARDict['prepped_aia']
    orbdirs = glob.glob(aiadir+'/*')
    orbdirs.sort()
    firstfiles = glob.glob(orbdirs[0]+'/*')
    firstfiles.sort()
    firstfile = firstfiles[0]
    lastfiles = glob.glob(orbdirs[-1]+'/*')
    lastfiles.sort()
    lastfile = lastfiles[-1] 
    print(firstfile, lastfile)
    with open(firstfile, 'rb') as f:
        fdata = pickle.load(f)
    with open(lastfile, 'rb') as f:
        ldata = pickle.load(f)

    fr = {'radius': 150.0,
          'centerx': fdata['centerx'],
          'centery': fdata['centery']
         }
    lr = {'radius': 150.0,
          'centerx': ldata['centerx'],
          'centery': ldata['centery']
         }
    regions = [ fr, lr]
    aagt = ARDict['per_region_all_time_intervals']
    
    aiamaps=[]
    for i in [0, -1]:
        try:
            #First region, orbit of interest, first time interval, start time of that interval
            start_time = aagt[0][i][0][0]
            print(start_time)
            
            jsoc_email='jessie.m.duncan@nasa.gov'
        
        
            #Get single full-disk 94 for region adjustment selection
            query = Fido.search(
                #a.Time(start_time+10*u.min, start_time + 11*u.min),
                a.Time(start_time, start_time + 10*u.s),
                a.Wavelength(94*u.angstrom),
                a.jsoc.Series.aia_lev1_euv_12s,
                a.jsoc.Notify(jsoc_email),
            )
            print(query)
            files = Fido.fetch(query)
        
            #Prep map
            amap=sunpy.map.Map(files[0])
            try:
                m = register(amap)
            except TypeError:
                amap.meta.pop('crpix1')
                amap.meta.pop('crpix2')
                print('CRPIX issue on ', files)
                m = register(amap)

            fig = plt.figure(figsize=(10,10))
            ax = fig.add_subplot(1,1,1, projection=m)
            m.plot()
            region=iac.make_region(regions[i], m)
            og_region = region.to_pixel(m.wcs)                    
            og_region.plot(axes=ax, color='blue', ls='--', lw=3, label='Chosen NuSTAR Region')
    
            aiamaps.append(m)
            
        except IndexError:
            aiamaps.append([])


In [ ]:
for nac in nccs_aia_cases:
    print(nac)
    ARDict = all_targets[nac]
    #print(ARDict.keys())
    aiadir = ARDict['prepped_aia']
    orbdirs = glob.glob(aiadir+'/*')
    orbdirs.sort()
    firstfiles = glob.glob(orbdirs[0]+'/*')
    firstfiles.sort()
    firstfile = firstfiles[0]
    lastfiles = glob.glob(orbdirs[-1]+'/*')
    lastfiles.sort()
    lastfile = lastfiles[-1] 
    #print(firstfile, lastfile)
    with open(firstfile, 'rb') as f:
        fdata = pickle.load(f)
    with open(lastfile, 'rb') as f:
        ldata = pickle.load(f)

    print(fdata['centerx'], fdata['centery'])
    print(ldata['centerx'], ldata['centery'])
    print('')

In [ ]:
for jac in jsoc_aia_cases:
    print(jac)
    ARDict = all_targets[jac]
    print(ARDict['nushift'])
    regions = ARDict['regdicts']
    print(regions)
    id_dirs = ARDict['datapaths']

    r = regions[0][0]
    print(r)
    #Get sample AIA files for each orbit!
    
    import sunpy.map
    from sunpy.net import Fido
    from sunpy.net import attrs as a
    
    from aiapy.calibrate.util import get_correction_table, get_pointing_table
    from aiapy.calibrate import register, update_pointing, degradation, estimate_error
    
    aagt = ARDict['per_region_all_time_intervals']
    
    aiamaps=[]
    for i in range(0, len(id_dirs)):
        try:
            #First region, orbit of interest, first time interval, start time of that interval
            start_time = aagt[0][i][0][0]
            print(start_time)
            
            jsoc_email='jessie.m.duncan@nasa.gov'
        
        
            #Get single full-disk 94 for region adjustment selection
            query = Fido.search(
                #a.Time(start_time+10*u.min, start_time + 11*u.min),
                a.Time(start_time, start_time + 10*u.s),
                a.Wavelength(94*u.angstrom),
                a.jsoc.Series.aia_lev1_euv_12s,
                a.jsoc.Notify(jsoc_email),
            )
            print(query)
            files = Fido.fetch(query)
        
            #Prep map
            amap=sunpy.map.Map(files[0])
            try:
                m = register(amap)
            except TypeError:
                amap.meta.pop('crpix1')
                amap.meta.pop('crpix2')
                print('CRPIX issue on ', files)
                m = register(amap)

            fig = plt.figure(figsize=(10,10))
            ax = fig.add_subplot(1,1,1, projection=m)
            m.plot()
            region=iac.make_region(r, m)
            og_region = region.to_pixel(m.wcs)                    
            og_region.plot(axes=ax, color='blue', ls='--', lw=3, label='Chosen NuSTAR Region')
    
            aiamaps.append(m)
            
        except IndexError:
            aiamaps.append([])


        


In [ ]:
for jac in jsoc_aia_cases:
    print(jac)
    ARDict = all_targets[jac]
    qfs = ARDict['res_file_dict(s)'][0]['quiet files all-inst']
    #print(qfs)

    all_aia_inputs=[]
    for f in qfs:
        with open(f, 'rb') as f:
            data = pickle.load(f)
        all_aia_inputs.append(np.array(data['dn_in'][0:7]))

    aai = np.array(all_aia_inputs)


    for i in range(0, 6):
        fig = plt.figure(figsize=(10,3))
        plt.plot(aai[:,i])
        plt.title(jac+' '+data['chanax'][i])